# Rupesh Man Pradhan GIS · V02.0 ward-summary map

**Parent frozen milestone:** [`rupesh-gis/v0.1.4-source`](https://github.com/FabinGurung/JP_Research-and-Development/releases/tag/rupesh-gis/v0.1.4-source)

Produces the regional **100 km buffer-derived square map extent**, a detailed Pokhara 33-ward inset, and qualified-ward engineered/non-engineered summaries. Does **not** plot any individual site coordinates. Use only on authorized private Drive data.

**Evidence boundary:** Kathmandu Living Labs / OpenStreetMap ward boundaries are historical (33 wards, area ~464.24 km²). GIS overlay is a **provisional geospatial cross-check**, *not* an official 2026 ward assignment. Compare against the researcher's recorded ward labels and do not overwrite either. One user-approved Ward 33 record remains counted under recorded Ward 33 even though the supplied coordinate is outside Pokhara. A reviewer must resolve all 50 source/polygon disagreements before thesis admission.

**Input:** Google Drive normalized CSV `1CZuvdqki26C7eTRFki9cslWuRTNactKh`, derived from the original, unchanged `GOOGLE CO ORDINATE.xlsx` (`1piJxk-V3xUl_CB0NxEnc5g1G2hbp1WK0`). Colab authenticates to Drive and downloads private bytes in memory; no personal site coordinates are published to GitHub.

**First use:** Runtime → Run all; Google sign-in / Drive authorization may be required. The outputs are created in `/content/RMP_GIS_V02_OUTPUTS` and optionally uploaded to existing Rupesh Drive folders using an idempotent file-name check.


In [ ]:
# 01 · Private source and vetted ward-boundary download
import io, os, csv, json, hashlib, urllib.request, pathlib, datetime, collections
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from shapely.geometry import shape, Point, mapping
from shapely.ops import unary_union, transform
from pyproj import Transformer
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon, FancyBboxPatch
import matplotlib.patheffects as pe
from IPython.display import display, HTML

SOURCE_DRIVE_CSV_ID = "1CZuvdqki26C7eTRFki9cslWuRTNactKh"
ORIGINAL_RESEARCHER_XLSX_ID = "1piJxk-V3xUl_CB0NxEnc5g1G2hbp1WK0"
WARD_SOURCE_URL = "https://raw.githubusercontent.com/KathmanduLivingLabs/prepare-pokhara-api-server/master/geojson-data/wards.json"
EXPECTED_GIT_BLOB_SHA1 = "4a8ca0c954aa8f492d3fddf00b17acee228eed3c"
BASELINE = "RMP_GIS_M01__V01_4_SOURCE_BASELINE__2026-10-12"
OUT = pathlib.Path('/content/RMP_GIS_V02_OUTPUTS')
OUT.mkdir(exist_ok=True)
auth.authenticate_user()
credentials,_ = default()
drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)

def drive_bytes(file_id):
    request = drive.files().get_media(fileId=file_id)
    fh = io.BytesIO(); downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done: _,done=downloader.next_chunk()
    return fh.getvalue()
raw = drive_bytes(SOURCE_DRIVE_CSV_ID)
rows = list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
assert len(rows) == 266 and sum(x['category']=='Engineered' for x in rows)==150 and sum(x['category']=='Non-engineered' for x in rows)==116, 'Do not continue on an unexpected research input'
with urllib.request.urlopen(WARD_SOURCE_URL, timeout=50) as response: boundary_bytes = response.read()
blob = hashlib.sha1(b'blob '+str(len(boundary_bytes)).encode()+b'\0'+boundary_bytes).hexdigest()
assert blob == EXPECTED_GIT_BLOB_SHA1, ('Boundary source changed; freeze, do not silently replace', blob)
ward_features=json.loads(boundary_bytes)['features']
assert len(ward_features)==33
full_wards={int(__import__('re').search(r'Ward No.\s*(\d+)',f['properties']['name']).group(1)):shape(f['geometry']) for f in ward_features}
assert sorted(full_wards)==list(range(1,34))
print('Source records: 266; engineered 150, non-engineered 116')
print('Ward polygons: 33; Git blob SHA-1',blob)


In [ ]:
# 02 · Spatial assignments using ORIGINAL/full-resolution polygons, never visually simplified polygons
proj = Transformer.from_crs('EPSG:4326','EPSG:32645',always_xy=True).transform
unproj = Transformer.from_crs('EPSG:32645','EPSG:4326',always_xy=True).transform
wards_m={w:transform(proj,p) for w,p in full_wards.items()}
all_poly=unary_union(list(wards_m.values()))

joined=[]
for r in rows:
    src=int(r['ward']); category=r['category']
    point=Point(float(r['longitude']),float(r['latitude']))
    hits=[w for w,p in full_wards.items() if p.covers(point)]
    if len(hits)==1:
        geo=hits[0]; status='SOURCE_SPATIAL_AGREE' if geo==src else 'SOURCE_SPATIAL_CONFLICT'; summary_ward=geo
    elif len(hits)==0 and r['point_id']=='RMP-ENG-W33-R207':
        geo=None;status='MANUAL_SOURCE_WARD33_APPROVED__GEOMETRY_OUTSIDE';summary_ward=33
    else:
        raise ValueError(f'Unreviewed nonunique/outside ward membership: {r["point_id"]}, hits={hits}')
    joined.append({'point_id':r['point_id'],'recorded_ward':src,'spatial_ward':geo,'summary_ward':summary_ward,'category':category,'source_row':r['source_row'],'status':status})
assert len(joined)==266
assert sum(r['status']=='SOURCE_SPATIAL_CONFLICT' for r in joined)==50
assert sum(r['status'].startswith('MANUAL_') for r in joined)==1
ward_summary=[]
for w in range(1,34):
    geo=[r for r in joined if r['summary_ward']==w]
    src=[r for r in joined if r['recorded_ward']==w]
    calc=lambda a,c:sum(r['category']==c for r in a)
    E,N=calc(geo,'Engineered'),calc(geo,'Non-engineered')
    e,n=calc(src,'Engineered'),calc(src,'Non-engineered')
    ward_summary.append({'ward':w,'spatial_engineered':E,'spatial_non_engineered':N,
        'recorded_engineered':e,'recorded_non_engineered':n,'spatial_both':E>0 and N>0,
        'recorded_both':e>0 and n>0,'source_spatial_disagreements':sum(r['status']=='SOURCE_SPATIAL_CONFLICT' and r['recorded_ward']==w for r in joined),
        'manual_location_exception':sum(r['status'].startswith('MANUAL_') and r['summary_ward']==w for r in joined)})
assert sum(r['spatial_both'] for r in ward_summary)==27
assert sum(r['spatial_engineered'] for r in ward_summary)==150
assert sum(r['spatial_non_engineered'] for r in ward_summary)==116

def write_csv(path,items):
    with open(path,'w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(items[0]));writer.writeheader();writer.writerows(items)
write_csv(OUT/'RMP_WARD_COUNTS_SPATIAL_VS_RECORDED_v0.2.0.csv',ward_summary)
write_csv(OUT/'RMP_WARD_SPATIAL_JOIN_QA_v0.2.0.csv',joined)
# Exact square extent in EPSG:32645, including a >=100 km margin past all the city's boundary bounds.
x0,y0,x1,y1=all_poly.bounds
cx,cy=(x0+x1)/2,(y0+y1)/2
half=max((x1-x0)/2,(y1-y0)/2)+100000
square=[cx-half,cy-half,cx+half,cy+half]
assert abs((square[2]-square[0])-(square[3]-square[1]))<1e-6
print('Qualifying wards:',[x['ward'] for x in ward_summary if x['spatial_both']])
print('Source/geometry disagreements: 50; user-approved Ward33 non-geometric match: 1')
print('Extent square side (km):',round(2*half/1000,2),'EPSG:32645')


In [ ]:
# 03 · Cartographically generalized shapes for display ONLY; map labels and a real projected-square extent
DRAW = {w:p.simplify(0.0012,preserve_topology=True) for w,p in full_wards.items()}
DRAW_M = {w:transform(proj,p) for w,p in DRAW.items()}
counts={x['ward']:x for x in ward_summary}
BLUE,ORANGE='#2262B8','#EC7936'
plt.rcParams['font.family']='DejaVu Sans'
fig=plt.figure(figsize=(19,12),facecolor='#F7F9FC')
fig.text(.04,.955,'RUPESH MAN PRADHAN  /  GIS RESEARCH',fontsize=12,color='#1A968E',weight='bold')
fig.text(.04,.916,'Pokhara · Ward-level building composition',fontsize=26,color='#13253C',weight='bold')
fig.text(.04,.882,'27 qualifying wards · source/geographic discrepancies preserved · no individual points',fontsize=12,color='#617588')
ax=fig.add_axes([.055,.41,.39,.445]);az=fig.add_axes([.49,.41,.455,.445])
for panel in (ax,az):
    panel.set_facecolor('#EDF4F8');panel.set_aspect('equal');panel.set_xticks([]);panel.set_yticks([])
    for k in panel.spines: panel.spines[k].set_visible(False)
ax.set_xlim(square[0],square[2]);ax.set_ylim(square[1],square[3]);
context=all_poly.bounds;pad=2500;az.set_xlim(context[0]-pad,context[2]+pad);az.set_ylim(context[1]-pad,context[3]+pad)
def add_polygons(panel,w,p,edge,lw=.65):
    for q in ([p] if p.geom_type=='Polygon' else p.geoms):
        xx,yy=q.exterior.xy
        panel.add_patch(MplPolygon(list(zip(xx,yy)),closed=True,facecolor='#C7E5EF' if counts[w]['spatial_both'] else '#E7EDF2',edgecolor=edge,linewidth=lw))
for w,p in DRAW_M.items(): add_polygons(ax,w,p,'#FFFFFF',.35);add_polygons(az,w,p,'#92AFC1',.8)
for w,p in DRAW_M.items():
    if p.area > 0.04e6:
        z=p.representative_point();az.text(z.x,z.y,str(w),ha='center',va='center',fontsize=8,color='#285573',weight='bold',path_effects=[__import__('matplotlib.patheffects',fromlist=['withStroke']).withStroke(linewidth=2.6,foreground='#fff')])
ax.text(.04,.96,'01   100 KM-BUFFERED SQUARE FRAME',transform=ax.transAxes,ha='left',va='top',fontsize=10,weight='bold',color='#14293d')
az.text(.04,.96,'02   CITY WARD DETAIL',transform=az.transAxes,ha='left',va='top',fontsize=10,weight='bold',color='#14293d')
fig.text(.04,.36,'WARD ANNOTATIONS  ·  BLUE ENGINEERED / ORANGE NON-ENGINEERED',fontsize=14,weight='bold',color='#14293d')
qual=[x for x in ward_summary if x['spatial_both']]
for i,r in enumerate(qual):
    col=i//9;idx=i%9;x=.04+col*.315;y=.33-idx*.0304
    fig.text(x,y,f'WARD {r["ward"]:02d}',fontsize=9,weight='bold',color='#19334a')
    fig.text(x+.117,y,f'E {r["spatial_engineered"]:02d}',fontsize=9,weight='bold',color=BLUE)
    fig.text(x+.185,y,f'NE {r["spatial_non_engineered"]:02d}',fontsize=9,weight='bold',color=ORANGE)
fig.text(.04,.026,'Kathmandu Living Labs / OSM 33-ward polygon geometry (historical)  ·  source research workbook  ·  QA review required for 50 conflicts',fontsize=9,color='#667a8d')
img=OUT/'RMP_POKHARA_QUALIFYING_WARDS_SUMMARY_v0.2.0.png'
fig.savefig(img,dpi=150,facecolor=fig.get_facecolor())
plt.close(fig)
print('Saved preview:',img)


In [ ]:
# 04 · Offline, self-contained HTML map: no external tile servers or CDN
# The underlying paths represent the REAL ward boundaries. The interactive legend is count-based, with no individual surveyed points.
import html
bbox=(x0-1500,y0-1500,x1+1500,y1+1500)
def svgpath(poly,extent,width,height):
    xmin,ymin,xmax,ymax=extent
    parts=[]
    for p in ([poly] if poly.geom_type=='Polygon' else poly.geoms):
        coords=list(p.exterior.coords)
        line=[]
        for j,(x,y) in enumerate(coords):
            X=(x-xmin)/(xmax-xmin)*width
            Y=height-(y-ymin)/(ymax-ymin)*height
            line.append(('M' if j==0 else 'L')+f'{X:.1f},{Y:.1f}')
        parts.append(' '.join(line)+' Z')
    return ' '.join(parts)
svgo=''.join(f'<path d="{svgpath(p,square,730,730)}" class="ward-shape" data-ward="{w}" />' for w,p in DRAW_M.items())
svgd=''.join(f'<path d="{svgpath(p,bbox,1150,730)}" class="ward-shape" data-ward="{w}" />' for w,p in DRAW_M.items())
labels=''.join(f'<text class="wardnum" x="{(p.representative_point().x-bbox[0])/(bbox[2]-bbox[0])*1150:.1f}" y="{730-(p.representative_point().y-bbox[1])/(bbox[3]-bbox[1])*730:.1f}" data-ward="{w}">{w}</text>' for w,p in DRAW_M.items())
DATA=ward_summary

HTML_TEMPLATE = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Rupesh GIS | Pokhara Ward Count Atlas</title>\n<style>:root{--navy:#17283d;--muted:#6a7c90;--blue:#2d6bc2;--orange:#e98142;--line:#dbe5ed;--bg:#f5f8fc}*{box-sizing:border-box}body{margin:0;font:15px system-ui,-apple-system,Segoe UI,sans-serif;background:var(--bg);color:var(--navy)}.shell{max-width:1760px;margin:auto;padding:22px}.headline{display:flex;align-items:end;justify-content:space-between;gap:20px;flex-wrap:wrap;margin-bottom:18px}.eyebrow{font-size:12px;letter-spacing:2px;color:#178d93;font-weight:800}h1{font-size:clamp(25px,3vw,40px);margin:7px 0 0}small,.muted{color:var(--muted)}.tag{border:1px solid #dbe5ed;border-radius:30px;padding:8px 14px;background:#fff;font-size:12px}.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px}.kpi,.pane,.tablepanel{background:white;border:1px solid var(--line);border-radius:16px;box-shadow:0 3px 12px #2140560a}.kpi{padding:17px}.kpi b{font-size:29px;display:block}.kpi span{color:var(--muted);font-size:12px}.toolbar{display:flex;align-items:center;justify-content:space-between;gap:15px;flex-wrap:wrap;padding:15px 17px;background:#fff;border:1px solid var(--line);border-radius:14px;margin-bottom:14px}.selector{display:flex;gap:12px;align-items:center;flex-wrap:wrap}select,button{background:#fff;border:1px solid #c7d5e2;border-radius:9px;padding:9px 12px;font:inherit;color:var(--navy);cursor:pointer}button.primary{background:#155a89;color:#fff;border-color:#155a89}.grids{display:grid;grid-template-columns:minmax(330px,5fr) minmax(530px,8fr);gap:16px}.pane{padding:15px;overflow:hidden}.pane h2{font-size:18px;margin:3px 0}.pane p{color:var(--muted);font-size:12px;margin:4px 0 12px}.mapbox{position:relative;border-radius:10px;overflow:hidden;background:#eef3f7;border:1px solid #e6eff5}.mapbox svg{width:100%;height:auto;display:block}.ward-shape{fill:#dae9f3;stroke:#98aaba;stroke-width:1.5;transition:fill .16s ease,stroke .16s ease;cursor:pointer}.ward-shape.nonqual{fill:#ebeff2;opacity:.55}.ward-shape:hover,.ward-shape.selected{fill:#56b5bd!important;stroke:#16314e;stroke-width:3}.context .ward-shape{stroke-width:.42}.wardnum{pointer-events:none;font-size:19px;font-weight:800;fill:#2b566d;paint-order:stroke;stroke:white;stroke-width:5;stroke-linejoin:round;text-anchor:middle}.wardnum.nonqual{fill:#9aabb7}.legend{display:flex;gap:17px;margin-top:12px;color:var(--muted);font-size:12px;flex-wrap:wrap}.swatch{width:10px;height:10px;border-radius:2px;display:inline-block;margin-right:5px}.panelhead{display:flex;justify-content:space-between;gap:15px;align-items:center}#count{font-size:12px;color:var(--muted)}.cards{display:grid;grid-template-columns:repeat(3,1fr);gap:10px}.wardcard{border:1px solid var(--line);background:#fff;border-radius:12px;padding:13px;cursor:pointer;text-align:left;transition:.16s}.wardcard:hover,.wardcard.active{border-color:#278da9;box-shadow:0 0 0 2px #c7eaf0}.wardcard b{display:block;margin-bottom:7px}.counts{display:flex;gap:17px;font-weight:800}.blue{color:var(--blue)}.orange{color:var(--orange)}.foot{font-size:12px;color:var(--muted);line-height:1.7;padding:15px 2px}#info{padding:12px 17px;border-left:4px solid #17918a;background:#fff;border-radius:8px;margin:16px 0;color:#2b465e;font-size:14px}.warning{border-left-color:#ce8652!important}.labelnote{position:absolute;bottom:10px;left:12px;padding:5px 8px;border-radius:8px;color:#39576c;background:#fffffff0;font-size:10px}.north{position:absolute;top:12px;right:14px;font-weight:800}#tooltip{position:fixed;pointer-events:none;background:#17324de9;color:#fff;padding:9px 12px;border-radius:10px;font-size:13px;display:none;z-index:10;max-width:250px;box-shadow:0 6px 20px #12223833}#tooltip b{color:#a7f4ef}@media(max-width:1080px){.grids{grid-template-columns:1fr}.kpis{grid-template-columns:repeat(2,1fr)}.cards{grid-template-columns:repeat(2,1fr)}}@media(max-width:600px){.cards{grid-template-columns:1fr}.shell{padding:10px}.kpis{gap:8px}.kpi b{font-size:23px}}</style></head><body><div class="shell"><header class="headline"><div><div class="eyebrow">RUPESH MAN PRADHAN  /  RESEARCH GIS</div><h1>Pokhara | Ward composition atlas</h1><div class="muted">Wards with both engineered and non-engineered buildings · counts only, no building markers</div></div><span class="tag">V02.0 · Review edition · based on V01.4 frozen milestone</span></header><div class="kpis"><div class="kpi"><span>Surveyed building records</span><b>266</b></div><div class="kpi"><span>Engineered</span><b style="color:var(--blue)">150</b></div><div class="kpi"><span>Non-engineered</span><b style="color:var(--orange)">116</b></div><div class="kpi"><span>Qualifying wards</span><b>27 / 33</b></div></div><div class="toolbar"><div class="selector"><b>Count basis</b><select id="mode"><option value="spatial">Spatial join (provisional)</option><option value="recorded">Original ward labels (audit)</option></select><label><input type="checkbox" id="both" checked> Both categories only</label></div><div class="selector"><button id="reset">Reset selection</button><span class="muted">Click a ward polygon or count card</span></div></div><div class="grids"><section class="pane"><h2>01 · Regional square map frame</h2><p>Pokhara boundary + 100 km outer margin · equal-sided UTM 45N projection</p><div class="mapbox context"><svg viewBox="0 0 730 730" xmlns="http://www.w3.org/2000/svg"><defs><pattern id="gr" width="56" height="56" patternUnits="userSpaceOnUse"><path d="M56 0H0V56" fill="none" stroke="#dbe6ee" stroke-width=".8"/></pattern></defs><rect width="730" height="730" fill="url(#gr)"/><g id="overview">__OVERVIEW__</g></svg><div class="north">↑ N</div><div class="labelnote">234.15 km × 234.15 km · EPSG:32645</div></div><div class="legend"><span><i class="swatch" style="background:#dae9f3"></i> Both categories</span><span><i class="swatch" style="background:#ebeff2"></i> Other wards</span><span><i class="swatch" style="background:#56b5bd"></i> Selected</span></div></section><section class="pane"><h2>02 · Pokhara ward detail</h2><p>Magnified to make ward boundaries legible · regional map frame stays fixed</p><div class="mapbox"><svg viewBox="0 0 1150 730" xmlns="http://www.w3.org/2000/svg"><g id="details">__DETAIL__</g><g id="wardnums">__NUMS__</g></svg><div class="labelnote">Ward numbers reference the count cards below</div></div></section></div><div id="info" role="status">Select any ward to view both counts, the recorded-source comparison, and QA discrepancies.</div><section class="tablepanel" style="padding:18px"><div class="panelhead"><h2>Ward-level engineered / non-engineered annotations</h2><span id="count"></span></div><div class="cards" id="cards"></div></section><footer class="foot">SOURCE: Kathmandu Living Labs / OpenStreetMap ward geometries (historical; not certified 2026), original Rupesh research workbook. Full-resolution polygon assignment is used for count QA; map vector lines are visually simplified (~130 m tolerance). <b>Review hold:</b> 50 source-ward/spatial-ward discrepancies; 1 Ward 33 user-approved source assignment not located by the recorded coordinates. Geographic survey CRS is provisional EPSG:4326. No individual location points are displayed.</footer></div><div id="tooltip"></div><script>const DATA=__DATA__;const items=new Map(DATA.map(x=>[x.ward,x]));const mode=document.querySelector(\'#mode\'),both=document.querySelector(\'#both\'),cards=document.querySelector(\'#cards\'),inf=document.querySelector(\'#info\'),cnt=document.querySelector(\'#count\'),tip=document.querySelector(\'#tooltip\');let selected=null;function getStats(x){return mode.value===\'spatial\'?[x.spatial_engineered,x.spatial_non_engineered]:[x.recorded_engineered,x.recorded_non_engineered]}function render(){let chosen=DATA.filter(x=>{let [a,b]=getStats(x);return !both.checked||(a>0&&b>0)});cnt.textContent=chosen.length+\' qualifying wards\';cards.innerHTML=chosen.map(x=>{let[a,b]=getStats(x);return `<button class="wardcard ${x.ward===selected?\'active\':\'\'}" data-ward="${x.ward}"><b>WARD ${String(x.ward).padStart(2,\'0\')} ${x.source_spatial_disagreements?\'†\':\'\'}</b><div class="counts"><span class="blue">E ${a}</span><span class="orange">NE ${b}</span></div></button>`}).join(\'\');for(let path of document.querySelectorAll(\'[data-ward].ward-shape\')){let x=items.get(+path.dataset.ward);let[a,b]=getStats(x);path.classList.toggle(\'nonqual\',both.checked&&!(a>0&&b>0));path.classList.toggle(\'selected\',selected===x.ward)}for(let el of document.querySelectorAll(\'.wardnum\')){let x=items.get(+el.dataset.ward),[a,b]=getStats(x);el.classList.toggle(\'nonqual\',both.checked&&!(a>0&&b>0));}if(selected){let x=items.get(selected),[e,n]=getStats(x);inf.className=\'warning\';inf.innerHTML=`<b>Ward ${selected}</b> — <span class="blue">engineered ${e}</span> / <span class="orange">non-engineered ${n}</span> · Source ward counts E ${x.recorded_engineered} / NE ${x.recorded_non_engineered} · Spatial counts E ${x.spatial_engineered} / NE ${x.spatial_non_engineered}. <b>${x.source_spatial_disagreements} source-label conflicts</b> originate from this ward.${x.manual_location_exception?\' Includes one user-approved Ward 33 location pending geometric verification.\':\'\'}`;}else{inf.className=\'\';inf.textContent=\'Select any ward to view both counts, original-source comparison and QA discrepancies.\'}}\ndocument.querySelectorAll(\'.ward-shape\').forEach(el=>{const w=+el.dataset.ward;el.addEventListener(\'click\',()=>{selected=w;render()});el.addEventListener(\'mousemove\',e=>{let x=items.get(w),[a,b]=getStats(x);tip.innerHTML=`<b>Ward ${w}</b><div>Engineered ${a} / Non-engineered ${b}</div><small>Source-label conflicts: ${x.source_spatial_disagreements}</small>`;tip.style.left=Math.min(e.clientX+12,innerWidth-260)+\'px\';tip.style.top=Math.min(e.clientY+14,innerHeight-100)+\'px\';tip.style.display=\'block\'});el.addEventListener(\'mouseleave\',()=>tip.style.display=\'none\')});cards.addEventListener(\'click\',e=>{const c=e.target.closest(\'.wardcard\');if(c){selected=+c.dataset.ward;render();document.querySelector(\'#info\').scrollIntoView({behavior:\'smooth\',block:\'center\'})}});document.querySelector(\'#reset\').addEventListener(\'click\',()=>{selected=null;render()});mode.addEventListener(\'change\',()=>{selected=null;render()});both.addEventListener(\'change\',render);render();</script></body></html>'

html_out=HTML_TEMPLATE.replace('__OVERVIEW__',svgo).replace('__DETAIL__',svgd).replace('__NUMS__',labels).replace('__DATA__',json.dumps(DATA,separators=(',',':')))
file=OUT/'RMP_POKHARA_WARD_SUMMARY_INTERACTIVE_v0.2.0.html'
file.write_text(html_out,encoding='utf-8')
# Show the HTML inside a Colab-hosted iframe without reloading external basemap services.
from google.colab import files
from IPython.display import IFrame
print('Saved offline HTML:', file)
display(IFrame(src='/files/'+OUT.name+'/'+file.name,width='100%',height=700))


In [ ]:
# 05 · Manifest, source–output lineage and safe optional Drive upload
manifest={
 'baseline':BASELINE,'edition':'v0.2.0-ward-summary-review',
 'source_csv_drive_id':SOURCE_DRIVE_CSV_ID,'original_workbook_drive_id':ORIGINAL_RESEARCHER_XLSX_ID,
 'boundary_source':WARD_SOURCE_URL,'boundary_git_blob_sha1':blob,
 'spatial_join':'FULL_33_WARD_ORIGINAL_POLYGONS','map_geometry':'GENERALIZED_DISPLAY_ONLY',
 'square_extent_utm45n_m':square,'square_side_km':round(2*half/1000,3),
 'status':'CANDIDATE_AWAITING_RESEARCHER_REVIEW',
 'qa':{'records':266,'engineered':150,'non_engineered':116,'qualifying_wards':27,'ward_label_disagreements':50,'user_approved_nonlocatable':1},
 'note':'Not a 2026 official ward polygon certification; CRS assumed WGS84; no private points published in Git.'}
manifest['file_sha256']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file()}
(OUT/'RMP_WARD_SUMMARY_RUN_MANIFEST_v0.2.0.json').write_text(json.dumps(manifest,indent=2))
print('Files:', [(p.name,round(p.stat().st_size/1024)) for p in OUT.iterdir() if p.is_file()])

# Optional upload: no blind overwrite, no duplication if exact named artifact already exists
UPLOAD_TO_DRIVE=False
MAPS_FOLDER_ID='1iAP-wSc4nSqjNW8odXaFg-ymQNNb46zj'
QA_FOLDER_ID='1sPcfq-LADyGnOZcYDK-5QyMpFKl1pWj6'
if UPLOAD_TO_DRIVE:
    for f in OUT.iterdir():
        if not f.is_file(): continue
        folder=MAPS_FOLDER_ID if f.suffix.lower() in ('.png','.html') else QA_FOLDER_ID
        safe_name=f.name.replace("'","\'")
        q=f"name = '{safe_name}' and '{folder}' in parents and trashed = false"
        prior=drive.files().list(q=q,fields='files(id,name)',page_size=10).execute()['files']
        if prior:
            print('Existing file; not overwritten:',f.name, prior[0]['id'])
            continue
        media=MediaFileUpload(str(f),resumable=True)
        obj=drive.files().create(body={'name':f.name,'parents':[folder]},media_body=media,fields='id,name,size').execute()
        print('Uploaded:',obj)
